# 03 — Final models on Kaggle

This notebook consumes the completed Colab CV selections and trains the selected TF-IDF baseline, RoBERTa, and DistilRoBERTa. It runs **no cross-validation search**.

1. Upload `thesis_ml_kaggle_inputs.zip` as a **private Kaggle Dataset**, then attach it with **Add Input**.
2. Import this notebook, enable **GPU** and **Internet**, and run the cells in order.
3. Save a notebook version with output files, and download the final experiment folder.

Settings are frozen: baseline C=1; RoBERTa LR=5e-5 for 6 epochs; DistilRoBERTa LR=2e-5 for 5 epochs; final seed=42. Both transformers retain the original six-epoch learning-rate schedule. The prior test set has already been used in preliminary experiments; report that limitation in the thesis.

Source: [Kaggle notebooks documentation](https://www.kaggle.com/docs/notebooks), [dataset uploads](https://www.kaggle.com/docs/datasets).

## Install dependencies
Run this cell first. If Kaggle requests a kernel restart, restart it and continue from the next cell. Internet is needed for package installation and the pinned pretrained model downloads. Keep Kaggle's supplied PyTorch.

In [ ]:
%pip install -q "transformers==4.57.1" "scikit-learn==1.6.1" "pandas>=2.2,<3" "matplotlib>=3.9,<4"

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.metadata, json, marshal, os, platform, random, re, time, types, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from IPython.display import display
LABELS = ['strumentale', 'intrinseco', 'ambiguo']
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
TEST_NOTE = ('The existing test set was already used in preliminary experiments. '
             'It is not a retrospectively untouched holdout.')
def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def safe_name(value):
    # Hugging Face IDs can contain a slash; local checkpoints can have absolute paths.
    return re.sub(r'[^A-Za-z0-9_.-]', '_', str(value)).strip('._')[:80] + '_' + json_hash(str(value))[:8]

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    os.replace(temporary, path)

def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def read_table(path, with_folds=False, allow_empty=False):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Copy the prepared CSV files from the original project.')
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    needed = {'id', 'thread_id', 'testo', 'etichetta'} | ({'fold'} if with_folds else set())
    if not needed.issubset(frame.columns):
        raise ValueError(f'{path.name}: missing columns {sorted(needed - set(frame.columns))}')
    if frame.empty and not allow_empty:
        raise ValueError(f'{path.name} is empty.')
    if frame['id'].duplicated().any():
        raise ValueError(f'{path.name}: duplicate example IDs.')
    for col in ['id', 'thread_id', 'testo']:
        if frame[col].str.strip().eq('').any():
            raise ValueError(f'{path.name}: blank {col}.')
    if not set(frame['etichetta']).issubset(LABELS):
        raise ValueError(f'{path.name}: unknown labels {set(frame.etichetta) - set(LABELS)}')
    if with_folds:
        numeric = pd.to_numeric(frame['fold'], errors='raise')
        if (numeric % 1 != 0).any():
            raise ValueError('Fold numbers must be integers.')
        frame['fold'] = numeric.astype(int)
        if set(frame.fold) != set(range(1, 6)):
            raise ValueError('Expected the five existing folds numbered 1 to 5.')
        if (frame.groupby('thread_id')['fold'].nunique() > 1).any():
            raise ValueError('A thread appears in more than one fold.')
    return frame

def load_development(data_dir):
    dev = read_table(data_dir / 'dev.csv', with_folds=True)
    supp = read_table(data_dir / 'supplementare.csv', allow_empty=True)
    if EXPECTED_DEV_ROWS is not None and len(dev) != EXPECTED_DEV_ROWS:
        raise ValueError(f'Expected {EXPECTED_DEV_ROWS} development rows, found {len(dev)}.')
    if set(dev.id) & set(supp.id):
        raise ValueError('Development and supplementary example IDs overlap.')
    return dev, supp

def metrics_for(truth, predictions):
    report = classification_report(truth, predictions, labels=LABELS,
                                   output_dict=True, zero_division=0)
    return {'macro_f1': float(report['macro avg']['f1-score']),
            'accuracy': float(np.mean(np.asarray(truth) == np.asarray(predictions))),
            'per_class': {label: report[label] for label in LABELS}}

def code_hash(functions):
    # Normalize notebook cell filenames and line positions, preserving executable code.
    def normalize(c):
        constants = tuple(normalize(x) if isinstance(x, types.CodeType) else x for x in c.co_consts)
        line_key = 'co_linetable' if hasattr(c, 'co_linetable') else 'co_lnotab'
        return c.replace(co_filename='', co_firstlineno=1, co_consts=constants, **{line_key: b''})
    return json_hash({f.__name__: hashlib.sha256(marshal.dumps(normalize(f.__code__))).hexdigest()
                      for f in functions})

def checked_test(dev, supp):
    # Only call in the final evaluation section, after selection is frozen.
    test = read_table(DATA_DIR / 'test.csv')
    if EXPECTED_TEST_ROWS is not None and len(test) != EXPECTED_TEST_ROWS:
        raise ValueError(f'Expected {EXPECTED_TEST_ROWS} test rows, found {len(test)}.')
    training = pd.concat([dev.drop(columns='fold'), supp], ignore_index=True)
    if set(training.id) & set(test.id) or set(training.thread_id) & set(test.thread_id):
        raise ValueError('Final training/test overlap in IDs or threads.')
    if set(training.etichetta) != set(LABELS):
        raise ValueError('Final training is missing a class.')
    return training, test

def show_confusion(truth, prediction, path):
    matrix = confusion_matrix(truth, prediction, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(3), LABELS, rotation=20)
    ax.set_yticks(range(3), LABELS)
    ax.set(xlabel='Predicted', ylabel='True')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    display(fig)
    plt.close(fig)

## Locate inputs and load frozen selections
The attached package is found automatically under `/kaggle/input`. `PACKAGE_DIR` is an optional override. Model outputs go to `/kaggle/working`; input files are never edited. `RUN_FINAL=True` makes this notebook ready to train. Set it to False to inspect the setup only.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/thesis_ml/results')
# Optional explicit path to the attached thesis_ml directory. Auto-detected by default.
PACKAGE_DIR = None  # Example: Path('/kaggle/input/thesis-ml-final/thesis_ml')
RUN_FINAL = True
# A session filter: keep both for the full comparison, or run one at a time.
ACTIVE_MODELS = ['roberta-base', 'distilroberta-base']
RUN_BASELINE = True
EXPECTED_DEV_ROWS = 641
EXPECTED_TEST_ROWS = 159

if PACKAGE_DIR is None:
    markers = sorted(INPUT_ROOT.rglob('bundle_manifest.json'))
    if len(markers) != 1:
        raise ValueError(f'Expected one thesis package, found {len(markers)}. '
                         'Attach thesis_ml_kaggle_inputs.zip as a Dataset or set PACKAGE_DIR explicitly.')
    PACKAGE_DIR = markers[0].parent
PACKAGE_DIR = Path(PACKAGE_DIR)
DATA_DIR = PACKAGE_DIR / 'data'
bundle = json.loads((PACKAGE_DIR / 'bundle_manifest.json').read_text())
for relative, expected in bundle['file_hashes'].items():
    path = PACKAGE_DIR / relative
    if not path.is_file() or file_hash(path) != expected:
        raise ValueError(f'Missing or changed uploaded file: {relative}')

source_dir = PACKAGE_DIR / bundle['transformer_directory']
baseline_dir = PACKAGE_DIR / bundle['baseline_directory']
CV_MANIFEST = json.loads((source_dir / 'manifest.json').read_text())
saved = json.loads((source_dir / 'selected_configs.json').read_text())
BASELINE_MANIFEST = json.loads((baseline_dir / 'manifest.json').read_text())
baseline_saved = json.loads((baseline_dir / 'selected_configs.json').read_text())
if saved['manifest_hash'] != json_hash(CV_MANIFEST) or not saved.get('complete'):
    raise ValueError('A complete selection matching the imported CV manifest is required.')
if baseline_saved['manifest_hash'] != json_hash(BASELINE_MANIFEST):
    raise ValueError('Baseline selection does not match its imported manifest.')
if baseline_saved['data_hashes'] != saved['data_hashes']:
    raise ValueError('Baseline and transformers used different development data.')
for name, expected in saved['data_hashes'].items():
    if file_hash(DATA_DIR / name) != expected:
        raise ValueError(f'{name} differs from the data used for CV.')

protocol = CV_MANIFEST['protocol']
MODEL_NAMES = protocol['models']
MAX_EPOCHS = protocol['max_epochs']  # Preserve the six-epoch LR schedule.
BATCH_SIZE = protocol['batch_size']
MAX_LENGTH = protocol['max_length']
WEIGHT_DECAY = protocol['weight_decay']
WARMUP_FRACTION = protocol['warmup_fraction']
USE_MIXED_PRECISION = protocol['mixed_precision']
FINAL_SEED = protocol['final_seed']
MODEL_REVISIONS = protocol['model_revisions']
if not set(ACTIVE_MODELS).issubset(MODEL_NAMES):
    raise ValueError('ACTIVE_MODELS must be a subset of the selected models.')
if saved['expected_models'] != MODEL_NAMES or set(saved['selected']) != set(MODEL_NAMES):
    raise ValueError('Missing model selections.')
for family, configuration in saved['selected'].items():
    if (configuration['family'] != family or
        configuration['learning_rate'] not in protocol['learning_rates'] or
        not 1 <= configuration['epoch'] <= MAX_EPOCHS):
        raise ValueError(f'Invalid frozen configuration for {family}.')
if BASELINE_MANIFEST['protocol']['seed'] != FINAL_SEED:
    raise ValueError('Baseline and transformer final seeds differ.')
if protocol['labels'] != LABELS or not protocol['weighted_loss']:
    raise ValueError('Unsupported imported training protocol.')
dev, supp = load_development(DATA_DIR)
print('Attached package:', PACKAGE_DIR)
display(pd.DataFrame([baseline_saved['overall_baseline']] + list(saved['selected'].values())))
print('Frozen final seed:', FINAL_SEED, '| Scheduler horizon:', MAX_EPOCHS, 'epochs')
print(TEST_NOTE)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
os.environ.setdefault('USE_TF', '0')
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

class EncodedComments(Dataset):
    def __init__(self, frame, tokenizer):
        self.encoded = tokenizer(frame.testo.tolist(), padding='max_length', truncation=True,
                                 max_length=MAX_LENGTH, return_tensors='pt')
        self.labels = torch.tensor([LABEL2ID[label] for label in frame.etichetta], dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {**{key: value[index] for key, value in self.encoded.items()}, 'labels': self.labels[index]}

@torch.no_grad()
def predict_transformer(model, loader, device):
    model.eval()
    predicted = []
    for batch in loader:
        inputs = {key: value.to(device) for (key, value) in batch.items() if key != 'labels'}
        logits = model(**inputs).logits
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
    return [LABELS[index] for index in predicted]

def train_transformer(model_name, learning_rate, seed, training, tokenizer,
                      epochs_to_train, validation=None, progress_dir=None):
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_ds = EncodedComments(training, tokenizer)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              generator=generator, num_workers=0)
    validation_loader = None
    if validation is not None:
        validation_loader = DataLoader(EncodedComments(validation, tokenizer),
                                       batch_size=BATCH_SIZE * 2, num_workers=0)
    counts = training.etichetta.value_counts()
    if any(counts.get(label, 0) == 0 for label in LABELS):
        raise ValueError('Training is missing a class.')
    weights = torch.tensor([len(training) / (3 * counts[label]) for label in LABELS],
                           dtype=torch.float32, device=device)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, revision=MODEL_REVISIONS.get(model_name), num_labels=3,
        id2label=dict(enumerate(LABELS)), label2id=LABEL2ID).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    # Always retain the six-epoch schedule, including shorter selected final training.
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer,
        num_warmup_steps=int(WARMUP_FRACTION * total_steps), num_training_steps=total_steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    amp_enabled = USE_MIXED_PRECISION and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    history, prediction_frames = [], []
    for epoch in range(1, epochs_to_train + 1):
        started = time.time()
        model.train()
        loss_sum = 0.0
        for batch in train_loader:
            y = batch.pop('labels').to(device)
            inputs = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(**inputs).logits
                loss = loss_fn(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f'Non-finite loss at epoch {epoch}. Inspect this run before changing the grid.')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()  # Do not advance after a skipped non-finite optimizer step.
            loss_sum += float(loss.detach().cpu())
        record = {'epoch': epoch, 'training_loss': loss_sum / len(train_loader),
                  'seconds': time.time() - started}
        if validation_loader is not None:
            prediction = predict_transformer(model, validation_loader, device)
            record.update(metrics_for(validation.etichetta.tolist(), prediction))
            frame = validation[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'}).copy()
            frame['epoch'], frame['predicted'] = epoch, prediction
            prediction_frames.append(frame)
        history.append(record)
        if progress_dir is not None:
            atomic_json(progress_dir / 'progress.json', {'status': 'partial', 'epochs': history,
                        'resume_note': 'If interrupted, this fold restarts; completed folds remain saved.'})
        score_text = f', macro-F1={record["macro_f1"]:.4f}' if 'macro_f1' in record else ''
        print(f'  epoch {epoch}/{epochs_to_train}, loss={record["training_loss"]:.4f}{score_text}, '
              f'{record["seconds"]:.0f}s', flush=True)
    predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else None
    return model, history, predictions

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
import joblib
def new_vectorizer():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000,
                          stop_words='english', sublinear_tf=True)

def new_classifier(family, C):
    if family == 'logistic_regression':
        return LogisticRegression(C=C, max_iter=10000, class_weight='balanced',
                                  solver='lbfgs', random_state=FINAL_SEED)
    if family == 'linear_svm':
        return LinearSVC(C=C, max_iter=10000, class_weight='balanced', dual=True,
                         random_state=FINAL_SEED)
    raise ValueError(f'Unknown baseline {family}')

def fit_baseline(family, C, X, y):
    model = new_classifier(family, C)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(X, y)
    convergence = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
    if convergence:
        raise RuntimeError('Classifier did not converge. Increase max_iter and rerun as a new experiment: '
                           + '; '.join(convergence))
    return model

## Record this final training environment
The original CV manifests remain intact. This final experiment records Kaggle's actual Python, packages, GPU, model revisions through the source manifest, selected settings, and data hashes. Completed models can be skipped when rerunning cells in the same experiment. An interrupted model starts again; changes to the environment create a separate final output directory.

In [ ]:
if ACTIVE_MODELS and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle notebook settings.')
DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
environment = {'python': platform.python_version(),
               'versions': {name: importlib.metadata.version(name) for name in
                            ['numpy', 'pandas', 'scikit-learn', 'matplotlib', 'torch', 'transformers']},
               'cuda': str(torch.version.cuda), 'gpu': DEVICE_NAME}
MANIFEST = {'workflow': 'frozen_cv_selection_final_training_v1',
            'source_cv_manifest_hash': json_hash(CV_MANIFEST),
            'source_baseline_manifest_hash': json_hash(BASELINE_MANIFEST),
            'transformer_selection': saved['selected'],
            'baseline_selection': baseline_saved['overall_baseline'],
            'data_hashes': {name: file_hash(DATA_DIR / name)
                            for name in ['dev.csv', 'supplementare.csv', 'test.csv']},
            'final_seed': FINAL_SEED, 'environment': environment,
            'training_code_hash': code_hash([seed_everything, EncodedComments.__init__,
                EncodedComments.__getitem__, predict_transformer.__wrapped__, train_transformer,
                new_vectorizer, new_classifier, fit_baseline, checked_test]),
            'test_note': TEST_NOTE}
EXP_DIR = OUTPUT_ROOT / ('kaggle_final_' + json_hash(MANIFEST)[:16])
EXP_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = EXP_DIR / 'manifest.json'
if manifest_path.exists() and json.loads(manifest_path.read_text()) != MANIFEST:
    raise ValueError('Existing final output manifest mismatch.')
atomic_json(manifest_path, MANIFEST)
atomic_json(EXP_DIR / 'source_cv_manifest.json', CV_MANIFEST)
atomic_json(EXP_DIR / 'source_baseline_manifest.json', BASELINE_MANIFEST)
atomic_json(EXP_DIR / 'selected_configs.json', saved)
atomic_json(EXP_DIR / 'baseline_selected_configs.json', baseline_saved)
print('Final output directory:', EXP_DIR)
print('Current environment:', environment)


In [ ]:
def assert_inputs_unchanged():
    for name, expected in MANIFEST['data_hashes'].items():
        if file_hash(DATA_DIR / name) != expected:
            raise ValueError(f'{name} changed during final training.')

def final_identity(configuration):
    return {'final_manifest_hash': json_hash(MANIFEST), 'selected': configuration,
            'test_hash': MANIFEST['data_hashes']['test.csv'], 'seed': FINAL_SEED,
            'test_note': TEST_NOTE}

def completed_final(directory, configuration):
    path = directory / 'evaluation.json'
    if not path.exists():
        return False
    result = json.loads(path.read_text())
    if result['identity'] != final_identity(configuration):
        raise ValueError(f'Existing final evaluation has different inputs/settings: {directory}')
    artifact = directory / ('model.joblib' if 'C' in configuration else 'checkpoint/config.json')
    if not artifact.is_file() or not (directory / 'test_predictions.csv').is_file():
        raise ValueError(f'Completed evaluation is missing its saved model/predictions: {directory}')
    print('Already completed:', directory.name)
    display(pd.DataFrame(result['report']).T)
    return True

def save_evaluation(directory, configuration, test, prediction, training_rows, history=None):
    assert_inputs_unchanged()
    report = classification_report(test.etichetta, prediction, labels=LABELS,
                                   output_dict=True, zero_division=0)
    predictions = test[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'}).copy()
    predictions['predicted'] = prediction
    atomic_csv(directory / 'test_predictions.csv', predictions)
    show_confusion(test.etichetta, prediction, directory / 'confusion_matrix.png')
    atomic_json(directory / 'evaluation.json', {'identity': final_identity(configuration),
                'report': report, 'training_rows': training_rows, 'test_rows': len(test),
                'training_history': history or [], 'device': DEVICE_NAME})
    display(pd.DataFrame(report).T)

if RUN_FINAL:
    assert_inputs_unchanged()
    training, test = checked_test(dev, supp)
    print(f'Final training: {len(training)} rows | Test: {len(test)} rows')
else:
    print('Final training disabled.')


## Train the selected baseline
TF-IDF is fitted on development plus supplementary training data. Test examples never enter training.

In [ ]:
if RUN_FINAL and RUN_BASELINE:
    configuration = baseline_saved['overall_baseline']
    directory = EXP_DIR / 'final' / configuration['family']
    if not completed_final(directory, configuration):
        directory.mkdir(parents=True, exist_ok=True)
        vectorizer = new_vectorizer()
        features = vectorizer.fit_transform(training.testo)
        classifier = fit_baseline(configuration['family'], configuration['C'], features, training.etichetta)
        prediction = classifier.predict(vectorizer.transform(test.testo))
        joblib.dump(Pipeline([('tfidf', vectorizer), ('classifier', classifier)]), directory / 'model.joblib')
        save_evaluation(directory, configuration, test, prediction, len(training))
        print('Baseline saved:', directory)


## Train the selected transformers
Each model starts from its pinned pretrained revision. Training duration comes from CV, and no checkpoint is selected using test scores. `ACTIVE_MODELS` can restrict this session to one model. This code uses one GPU, even if Kaggle supplies two.

In [ ]:
if RUN_FINAL:
    for model_name in ACTIVE_MODELS:
        assert_inputs_unchanged()
        configuration = saved['selected'][model_name]
        directory = EXP_DIR / 'final' / safe_name(model_name)
        if completed_final(directory, configuration):
            continue
        directory.mkdir(parents=True, exist_ok=True)
        tokenizer = AutoTokenizer.from_pretrained(model_name, revision=MODEL_REVISIONS[model_name])
        model, history, _ = train_transformer(model_name, configuration['learning_rate'],
            FINAL_SEED, training, tokenizer, configuration['epoch'], progress_dir=directory)
        checkpoint = directory / 'checkpoint'
        model.save_pretrained(checkpoint)
        tokenizer.save_pretrained(checkpoint)
        loader = DataLoader(EncodedComments(test, tokenizer), batch_size=BATCH_SIZE * 2)
        prediction = predict_transformer(model, loader, next(model.parameters()).device)
        save_evaluation(directory, configuration, test, prediction, len(training), history)
        print('Checkpoint:', checkpoint)
        del model, tokenizer, loader
        gc.collect()
        torch.cuda.empty_cache()


## Compare and save outputs
The comparison reports the frozen models on the same test examples. It does not alter the CV selections. Each model directory contains predictions, per-class metrics, a confusion matrix, and the saved model. Transformer `checkpoint/` directories contain both tokenizer and model files for subsequent inference.

Save a Kaggle notebook version with its output files and download the experiment directory. To continue in a fresh session, previous saved notebook outputs can be attached with **Add Input** and copied into `/kaggle/working/thesis_ml/results/` before the provenance cell; completed models are skipped if their final manifest still matches.

In [ ]:
rows = []
for path in sorted((EXP_DIR / 'final').glob('*/evaluation.json')):
    result = json.loads(path.read_text())
    if result['identity']['final_manifest_hash'] != json_hash(MANIFEST):
        raise ValueError(f'Incompatible final evaluation: {path}')
    row = {'model': result['identity']['selected']['family'],
           'macro_f1': result['report']['macro avg']['f1-score'],
           'accuracy': result['report']['accuracy']}
    for label in LABELS:
        row[f'{label}_f1'] = result['report'][label]['f1-score']
    rows.append(row)
if rows:
    comparison = pd.DataFrame(rows)
    atomic_csv(EXP_DIR / 'final_comparison.csv', comparison)
    display(comparison)
expected = {baseline_saved['overall_baseline']['family'], *MODEL_NAMES}
print('Comparison complete:', {row['model'] for row in rows} == expected)
print('Save this notebook version with its output files before ending the session.')
print('Download the experiment folder from Kaggle Output:', EXP_DIR)
